# CIC-IDS2018 Thursday Webattack: NFStream extraction and labeling

This notebook extracts and consolidates the PCAP capture parts associated with Cic2018 Thursday Webattack, applies the original attack-labeling rules, labels remaining flows as benign, removes duplicates, and exports the CSV consumed by the CIC-IDS2018 consolidation notebook. Only paths and filenames in the configuration cell should be changed.

## 1. Configuration

In [ ]:
from pathlib import Path

PCAP_DIR = Path("/path/to/cic-ids2018/pcaps")
OUTPUT_DIR = Path("/path/to/output/nfstream_daily_csv")
OUTPUT_FILE = OUTPUT_DIR / "06_cic2018_thursday_nfstream_webattack_22_02_18.csv"
PCAP_FILES = [
    "UCAP172.31.69.28",
]

IDLE_TIMEOUT = 300
ACTIVE_TIMEOUT = 20
STATISTICAL_ANALYSIS = True
DECODE_TUNNELS = True
BPF_FILTER = 'ip'
TIMEZONE = "America/Moncton"
DROP_COLUMNS = ["content_type", "user_agent", "server_fingerprint", "client_fingerprint", "requested_server_name"]

## 2. Imports and helper functions

In [ ]:
import pandas as pd
import pytz
import nfstream
from nfstream import NFStreamer

def validate_inputs():
    if not PCAP_DIR.is_dir(): raise NotADirectoryError(f"PCAP directory not found: {PCAP_DIR}")
    missing=[PCAP_DIR/name for name in PCAP_FILES if not (PCAP_DIR/name).is_file()]
    if missing: raise FileNotFoundError("Missing PCAP files:\n"+"\n".join(f"- {p}" for p in missing))

def extract_all():
    frames=[]
    for position,name in enumerate(PCAP_FILES,1):
        print(f"[{position}/{len(PCAP_FILES)}] Extracting {name}")
        frame=NFStreamer(source=str(PCAP_DIR/name),idle_timeout=IDLE_TIMEOUT,active_timeout=ACTIVE_TIMEOUT,statistical_analysis=STATISTICAL_ANALYSIS,decode_tunnels=DECODE_TUNNELS,bpf_filter=BPF_FILTER).to_pandas()
        frames.append(frame)
        print(f"    Extracted flows: {len(frame):,}")
    return pd.concat(frames,ignore_index=True)

def prepare(frame):
    result=frame.drop(columns=DROP_COLUMNS,errors="ignore").dropna().copy()
    timestamps=pd.to_datetime(result["src2dst_first_seen_ms"],unit="ms",utc=True).dt.tz_convert(pytz.timezone(TIMEZONE))
    result["Timestamp"]=timestamps.dt.strftime("%d/%m/%Y %I:%M")
    result["label"]=""
    result["multiclass"]=""
    return result.reset_index(drop=True)

def summary(frame,column):
    return pd.DataFrame({"count":frame[column].value_counts(),"percentage":frame[column].value_counts(normalize=True).mul(100).round(2)})

## 3. Validate and extract the PCAP files

In [ ]:
print(f"NFStream version: {nfstream.__version__}")
validate_inputs()
raw_flows=extract_all()
print(f"Total extracted flows: {len(raw_flows):,}")

## 4. Clean and prepare the extracted flows

In [ ]:
df=prepare(raw_flows)
print(f"Flows after cleaning: {len(df):,}")

## 5. Apply attack-specific labeling rules

In [ ]:
df.loc[(df["dst_ip"] == '172.31.69.28') &
                                  (df["dst_port"] == 80) &
                                  (df["protocol"] == 6) &
                                  (df['Timestamp'].between('22/02/2018 10:17','22/02/2018 11:24')),
                                  df.multiclass.name] = 'webattack_bruteforce'

df.loc[(df["dst_ip"] == '172.31.69.28') &
                                  (df["dst_port"] == 80) &
                                  (df["protocol"] == 6) &
                                  (df['Timestamp'].between('22/02/2018 04:15','22/02/2018 04:29')),
                                  df.multiclass.name] = 'webattack_sql_injection'

df.loc[(df["dst_ip"] == '172.31.69.28') &
                                  (df["dst_port"] == 80) &
                                  (df["protocol"] == 6) &
                                  (df['Timestamp'].between('22/02/2018 01:50','22/02/2018 02:29')),
                                  df.multiclass.name] = 'webattack_xss'

df.loc[df["multiclass"].eq(""),"multiclass"]="benign"
df["label"]=df["multiclass"].map(lambda value: "benign" if value=="benign" else "malign")

## 6. Remove duplicates and summarize

In [ ]:
rows_before=len(df)
df=df.drop_duplicates().reset_index(drop=True)
if df[["label","multiclass"]].eq("").any(axis=1).any(): raise ValueError("Unlabeled flows remain.")
print(f"Duplicate rows removed: {rows_before-len(df):,}")
display(summary(df,"label"))
display(summary(df,"multiclass"))

## 7. Export

In [ ]:
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
df.to_csv(OUTPUT_FILE,index=False)
print(f"Saved {len(df):,} flows to {OUTPUT_FILE.resolve()}")